In [1]:
import cadquery as cq
from cadquery import exporters
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Girt", cad_width=800, height=600)
set_default_viewer("Girt")

In [3]:
# All dimensions in inches.

POST_W = 4.0       # X dimension
POST_D = 6.0       # Y dimension
POST_H = 48.0      # Z dimension for visualization

GIRT_W = 6.0       # Z dimension
GIRT_D = 4.0       # Y dimension
GIRT_L = 36.0      # X dimension for visualization

TENON_T = 1.5      # tenon thickness in Y
TENON_H = 6.0      # tenon height in Z
TENON_L = 4.0      # passes completely through 4" post

JOINT_Z = 24.0     # centerline of girt/post connection
DROP_JOINT_Z = 36.0 # centerline of drop girt/post connection

In [4]:
post = (
    cq.Workplane("XY")
    .box(POST_W, POST_D, POST_H)
    .translate((0, 0, POST_H / 2))
)

In [5]:
mortise_a = (
    cq.Workplane("XY")
    .box(
        TENON_T,          # X: tenon thickness
        POST_D + 0.2,     # Y: through 6" post dimension
        TENON_H           # Z
    )
    .translate((0, 0, DROP_JOINT_Z))
)

mortise_b = (
    cq.Workplane("XY")
    .box(
        POST_W + 0.2,     # X: through 4" post dimension
        TENON_T,          # Y: tenon thickness
        TENON_H           # Z
    )
    .translate((0, 0, JOINT_Z))
)

post_with_mortise = post.cut(mortise_a).cut(mortise_b)

In [6]:
post_near_face_x = -POST_W / 2

girt_body = (
    cq.Workplane("XY")
    .box(GIRT_L, GIRT_D, GIRT_W)
    .translate((
        post_near_face_x - GIRT_L / 2,
        0,
        JOINT_Z
    ))
)

In [7]:
tenon = (
    cq.Workplane("XY")
    .box(
        TENON_L,
        TENON_T,
        TENON_H
    )
    .translate((
        post_near_face_x + TENON_L / 2,
        0,
        JOINT_Z
    ))
)

In [8]:
# Girt body extends away from the +X face
post_near_face_y = (-POST_D / 2)

drop_girt_body = (
    cq.Workplane("XY")
    .box(
        GIRT_D,   # length along X
        GIRT_L,   # 6" depth along Y
        GIRT_W    # 4" vertical
    )
    .translate((
        0,
        post_near_face_y - GIRT_L/2,
        DROP_JOINT_Z
    ))
)

# Tenon extends from the girt back through the 6" post
drop_tenon = (
    cq.Workplane("XY")
    .box(
        TENON_T,  # 1.5" thickness
        POST_D,   # 6" through-tenon length
        TENON_H
    )
    .translate((
        0,        # centered through post in X
        0,
        DROP_JOINT_Z
    ))
)

In [9]:
girt_with_tenon = girt_body.union(tenon).clean()
drop_girt = drop_girt_body.union(drop_tenon).clean()

In [10]:
#show_object(post_with_mortise, name="post")
#show_object(girt_with_tenon, name="girt")

In [11]:
joint = cq.Assembly()

joint.add(
    post_with_mortise,
    name="post"
)

joint.add(
    girt_with_tenon,
    name="girt"
)

joint.add(drop_girt, name="drop_girt")

show_object(joint)

+++
